In [ ]:
import os
from llama_parse import LlamaParse
from langchain_core.documents import Document as LC_Document
from langchain_experimental.text_splitter import SemanticChunker
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
import numpy as np
import weaviate
from weaviate.classes.config import Configure, Property, DataType

from sentence_transformers import CrossEncoder
from langchain_ollama import OllamaLLM
import json

In [78]:
from dotenv import load_dotenv

load_dotenv()
api_lp_key = os.getenv("LLAMA_CLOUD_API_KEY")
api_key = os.getenv("OPENAI_API_KEY")

Load the Data and Parsing the data

In [33]:
parser = LlamaParse(
    api_key = api_lp_key,
    result_type = "markdown", # For simple documents you can text instead of markdown. Markdown is more reliable when reading the tables and images
    num_workers = 2,
    verbose = True,
    language = "en"
)

file_path = "./file_survey_paper.pdf"
documents = parser.load_data(file_path)


Started parsing the file under job_id f330d26a-0fad-4b8f-b54b-025d7cee0aa0


c:\Users\jeeva\OneDrive\Desktop\C_projects\NLP\.venv\Lib\site-packages\pydantic\main.py:250: ResourceWarning: unclosed <socket.socket fd=3848, family=2, type=1, proto=0, laddr=('127.0.0.1', 51838), raddr=('127.0.0.1', 11434)>
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)


In [34]:
print(f"Loaded {len(documents)} document pages/sections.")
print(f"Sample Content:\n{documents[0].text[:500]}...")

Loaded 144 document pages/sections.
Sample Content:
arXiv:2303.18223v19 [cs.CL] 18 Mar 2026

# A Survey of Large Language Models

Wayne Xin Zhao, Kun Zhou*, Junyi Li*, Tianyi Tang, Xiaolei Wang, Yupeng Hou, Yingqian Min, Beichen Zhang, Junjie Zhang, Zican Dong, Yifan Du, Chen Yang, Yushuo Chen, Zhipeng Chen, Jinhao Jiang, Ruiyang Ren, Yifan Li, Xinyu Tang, Zikang Liu, Peiyu Liu, Jian-Yun Nie and Ji-Rong Wen†

# Abstract

Ever since the Turing Test was proposed in the 1950s, humans have explored the mastering of language intelligence by machine. L...


Using recursive splitting and Semantic chunking

In [ ]:
embed_model = OllamaEmbeddings(model="nomic-embed-text")

# Convert LlamaIndex Documents → LangChain Documents with explicit metadata mapping
lc_documents = []
for doc in documents:
    # LlamaParse usually uses 'page_label' for the human-readable page number
    page_num = doc.metadata.get("page_label", "Unknown")
    
    lc_documents.append(
        LC_Document(
            page_content=doc.text,
            metadata={
                "source": doc.metadata.get("file_name", "Unknown"),
                "page_number": page_num  # Explicitly store it here
            }
        )
    )

# Before doing the semantic chunking. I am doing additional chunking. so, a chunk can't have more than 8192 tokens. which is maximum for model.
pre_splitter = RecursiveCharacterTextSplitter(
    chunk_size=4000,       
    chunk_overlap=400,     # small overlap to preserve context at boundaries
)
pre_split_docs = pre_splitter.split_documents(lc_documents)
print(f"Pre-split into {len(pre_split_docs)} chunks before semantic chunking.")


chunks = SemanticChunker(
    embed_model,
    breakpoint_threshold_type="percentile", 
    breakpoint_threshold_amount=95.0         # tune: lower = more chunks, higher = fewer
)


semantic_chunks = chunks.split_documents(pre_split_docs)

Pre-split into 298 chunks before semantic chunking.


In [36]:
print(f"Created {len(semantic_chunks)} semantic chunks using nomic-embed-text.")
print(f"Example Chunk Content:\n{semantic_chunks[0].page_content[:200]}...")

Created 1108 semantic chunks using nomic-embed-text.
Example Chunk Content:
arXiv:2303.18223v19 [cs.CL] 18 Mar 2026

# A Survey of Large Language Models

Wayne Xin Zhao, Kun Zhou*, Junyi Li*, Tianyi Tang, Xiaolei Wang, Yupeng Hou, Yingqian Min, Beichen Zhang, Junjie Zhang, Zi...


Generating Embeddings and Using the Matryoshka Representation Learning(MRL) 

In [37]:
#We are using MRL to reduce the dimension of embeddings (To save space in the database and easy for search and to retreive the chunks from database)

Target_dim = 256

processed_chunks = []
print(f"Slicing vectors to {Target_dim} dimensions using MRL")

for chunk in semantic_chunks:
    full_vector = embed_model.embed_query(chunk.page_content)
    #we pulling the model from the Ollama to generate the embeddings
    sliced_vector = np.array(full_vector[:Target_dim])
    #MRL slicing. reducing from standard dimension to 256 dimension

    #we have to do renormalization for sliced vectors
    norm = np.linalg.norm(sliced_vector)
    if norm > 0:
        normalized_vector = sliced_vector / norm
    else:
        normalized_vector = sliced_vector

    #store the result with its metadata
    processed_chunks.append({
        "content": chunk.page_content,
        "metadata": chunk.metadata,
        "vector": normalized_vector.tolist()
    })


Slicing vectors to 256 dimensions using MRL


In [38]:
print(f"Ready for Weaviate: {len(processed_chunks)} vectors at {Target_dim} dims.")

Ready for Weaviate: 1108 vectors at 256 dims.


In [40]:
# 1. Connect to Weaviate
client = weaviate.connect_to_local()

try:
    collection_name = "LlamaParse_MRL_nomic"

    # Re-create the collection to ensure fresh metadata schema
    if client.collections.exists(collection_name):
        client.collections.delete(collection_name)
    
    client.collections.create(
        name=collection_name,
        vectorizer_config=Configure.Vectorizer.none(), # Manual MRL vectors
        properties=[
            Property(name="content", data_type=DataType.TEXT),
            Property(name="source", data_type=DataType.TEXT),
            Property(name="page_number", data_type=DataType.TEXT), # Changed to TEXT to handle 'page_label'
        ]
    )
    
    collection = client.collections.get(collection_name)

    # 2. Batch Upload with Metadata Mapping
    print(f"Starting ingestion of {len(semantic_chunks)} chunks...")
    
    with collection.batch.dynamic() as batch:
        for i, chunk in enumerate(semantic_chunks):
            # A. Get full embedding from Ollama
            full_vector = embed_model.embed_query(chunk.page_content)
            
            # B. MRL Slice (256-dim) & Normalize
            sliced_vec = np.array(full_vector[:256])
            norm = np.linalg.norm(sliced_vec)
            final_vec = (sliced_vec / norm).tolist() if norm > 0 else sliced_vec.tolist()

            # C. Extract correct Page Label (Handles LlamaParse specific metadata)
            page_val = str(chunk.metadata.get("page_label", chunk.metadata.get("page_number", "Unknown")))

            batch.add_object(
                properties={
                    "content": chunk.page_content,
                    "source": str(chunk.metadata.get("file_name", "Unknown")),
                    "page_number": page_val
                },
                vector=final_vec
            )

    print(f"\n SUCCESS: Ingested {len(semantic_chunks)} chunks with Page Citations.")

finally:
    client.close()

c:\Users\jeeva\OneDrive\Desktop\C_projects\NLP\.venv\Lib\site-packages\weaviate\warnings.py:196: DeprecationWarning: Dep024: You are using the `vectorizer_config` argument in `collection.config.create()`, which is deprecated.
            Use the `vector_config` argument instead.
            
  warnings.warn(


Starting ingestion of 1108 chunks...

 SUCCESS: Ingested 1108 chunks with Page Citations.


## Retrieval Pipeline

User Query and Converting into Embedding and using MRL

In [41]:
def user_query_embed():
    user_query = input("Type here")

    print("Generating embedding...")
    full_vector = embed_model.embed_query(user_query)

    #Applying the MRL
    target_dim = 256
    sliced_vector = np.array(full_vector[:target_dim])

    #Renormalization
    norm = np.linalg.norm(sliced_vector)

    if norm>0:
        normalized_query_vector = (sliced_vector/norm).tolist()
    else:
        normalized_query_vector = sliced_vector.tolist()

    print(f"User query converted into {len(normalized_query_vector)} dimensions")

    return user_query, normalized_query_vector

#Executing the step 1
query_text, query_vector = user_query_embed()


Generating embedding...
User query converted into 256 dimensions


Vector Search/Similarity Search

In [42]:
def vector_search(query_vec):
    top_k = int(input("Number of chunks should we retrive for context?"))

    # connect to weaviate
    client = weaviate.connect_to_local()

    try:
        #Accessing the MRL collection
        collection = client.collections.get("LlamaParse_MRL_nomic")

        print(f"Searching weaviate for top {top_k} matches...")\
        # perform the Near-vector search
        response  = collection.query.near_vector(
            near_vector = query_vec,
            limit = top_k,
            return_properties = ["content", "page_number", "source"]
        )
        
        # Extract results
        retrieved_results = []
        for obj in response.objects:
            retrieved_results.append(
                {
                    "content": obj.properties["content"],
                    "page": obj.properties["page_number"]
                }
            )

        print(f"Sucessfully retrieved {len(retrieved_results)} chunks.")

        return retrieved_results
        
    finally:
        client.close()

# Executing the step 2
retrieved_chunks = vector_search(query_vector)

# preview
for i, chunk in enumerate(retrieved_chunks):
    print(f"\n---Result {i+1} (page {chunk['page']})---")
    print(f"{chunk['content'][:200]}....")


Searching weaviate for top 50 matches...
Sucessfully retrieved 50 chunks.

---Result 1 (page Unknown)---
# given an increase in compute budget, the KM scaling law favors a larger budget allocation in model size than the data size, while the Chinchilla scaling law argues that the two sizes should be incre....

---Result 2 (page Unknown)---
As another representative study, Hoffmann et al. [34] (the Google DeepMind team) proposed an alternative form for scaling laws to instruct the optimal training for LLMs. They conducted rigorous experi....

---Result 3 (page Unknown)---
After introducing the formulations, we continue to discuss scaling law in the following two aspects, to enhance its understanding:

- Predictable scaling. In practice, scaling law can be used to instr....

---Result 4 (page Unknown)---
Since there was not a model trained following this law in the original paper, we took the last names of the two co-first authors to name this scaling law. 6. Here, Nc, Dc and Cc are measu

Re-Ranking the chunks


In [44]:
from sentence_transformers import CrossEncoder

def perform_reranking(query, retrieved_chunks):
    print("Initializing Cross-Encoder for re-ranking...")
    # 'cross-encoder/ms-marco-MiniLM-L-6-v2' is fast and highly effective for RAG
    reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

    # Prepare pairs: [(Query, Chunk1), (Query, Chunk2), ...]
    pairs = [[query, chunk['content']] for chunk in retrieved_chunks]
    
    # Compute Relevance Scores
    scores = reranker.predict(pairs)
    
    # Attach scores to chunks and Sort
    for i, score in enumerate(scores):
        retrieved_chunks[i]['rerank_score'] = score
        
    # Sort by the new re-ranker score (highest first)
    reranked_chunks = sorted(retrieved_chunks, key=lambda x: x['rerank_score'], reverse=True)
    
    print(f"Re-ranking complete. Most relevant chunk score: {reranked_chunks[0]['rerank_score']:.4f}")
    return reranked_chunks

# Execute Step 3
reranked_chunks = perform_reranking(query_text, retrieved_chunks)

# View the new order
for i, chunk in enumerate(reranked_chunks[:3]):
    #print(f"\n--- Rank {i+1} (Original FAISS Rank was unknown) ---")
    print(f"Page: {chunk['page']} | Re-rank Score: {chunk['rerank_score']:.4f}")
    print(f"{chunk['content'][:150]}...")

Initializing Cross-Encoder for re-ranking...


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 4216.67it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Re-ranking complete. Most relevant chunk score: 7.0682
Page: Unknown | Re-rank Score: 7.0682
# given an increase in compute budget, the KM scaling law favors a larger budget allocation in model size than the data size, while the Chinchilla sca...
Page: Unknown | Re-rank Score: -0.6334
For this survey, we also create a GitHub project website by collecting the supporting resources for LLMs, at the link https://github.com/RUCAIBox/LLMS...
Page: Unknown | Re-rank Score: -1.2823
As another representative study, Hoffmann et al. [34] (the Google DeepMind team) proposed an alternative form for scaling laws to instruct the optimal...


Filtering the retrieved chunks to 5 chunks

In [45]:

def distill_context(query, retrieved_50_chunks):
    # Using a fast, efficient model for filtering
    filter_llm = OllamaLLM(model="gemma3:4b") 

    print(f"Distilling 50 chunks down to the 5 most relevant...")

    # Prepare a list of chunks with IDs for the model to pick from
    chunk_list_text = ""
    for i, chunk in enumerate(retrieved_50_chunks):
        chunk_list_text += f"ID {i}: [Page {chunk['page']}] {chunk['content'][:300]}...\n\n"

    distill_prompt = f"""
    You are a context filterer. I will give you a USER QUERY and 50 snippets from a document.
    Your task is to identify the IDs of the 5 snippets that are most essential to answering the query.

    USER QUERY: {query}

    SNIPPETS:
    {chunk_list_text}

    Return ONLY a JSON list of the 5 integer IDs, for example: [1, 5, 22, 40, 49]
    """

    # 3. Get the IDs from the filter model
    raw_output = filter_llm.invoke(distill_prompt)
    
    try:
        # Extract the list from the string (handling potential LLM chatter)
        start = raw_output.find('[')
        end = raw_output.find(']') + 1
        best_ids = json.loads(raw_output[start:end])
        
        # 4. Filter the original list to just the 5 best
        distilled_chunks = [retrieved_50_chunks[idx] for idx in best_ids if idx < len(retrieved_50_chunks)]
        
        print(f"Distillation complete. Kept {len(distilled_chunks)} chunks.")
        return distilled_chunks[:5] # Safety slice
    except Exception as e:
        print(f" Distillation failed, falling back to top 5: {e}")
        return retrieved_50_chunks[:5]

Prompt Synthesis

In [47]:
def synthesize_final_prompt(query, top_chunks):
    print("Synthesizing the final grounded prompt...")
    
    # Format the context from the top 5 chunks
    top_context = ""
    for i, chunk in enumerate(top_chunks[:5]):
        top_context += f"\n--- CHUNK {i+1} (SOURCE: PAGE {chunk['page']}) ---\n"
        top_context += f"{chunk['content']}\n"

    # Build the structural prompt with strict rules
    system_rules = (
        "You are a professional research assistant. Use ONLY the provided context to answer.\n"
        "If the answer is missing, say you don't know. Always cite [Page X] for every fact."
    )
    
    final_prompt = f"""{system_rules}

CONTEXT:
{top_context}

USER QUESTION: 
{query}

GROUNDED ANSWER:"""

    return final_prompt

# Run Step 4
final_prompt_text = synthesize_final_prompt(query_text, reranked_chunks)

#print(final_prompt_text)

Synthesizing the final grounded prompt...


LLM Generation

In [48]:
def run_deepseek_generation(prompt):
    # Initialize the reasoning engine
    # DeepSeek-R1 is excellent for following the "Use ONLY context" rule
    llm = OllamaLLM(model="deepseek-r1:8b") 

    print("DeepSeek-R1 is thinking and generating citations...")
    
    # Generate Response
    response = llm.invoke(prompt)
    return response

# Run Step 5
final_answer = run_deepseek_generation(final_prompt_text)

print("\n" + "="*60)
print("FINAL RESPONSE:")
print(final_answer)
print("="*60)

DeepSeek-R1 is thinking and generating citations...

FINAL RESPONSE:
The KM scaling law (Kaplan et al. [30]) models performance as separate power-law relationships for model size (N), dataset size (D), and compute (C), each with distinct coefficients (αN, αD, αC). This approach does not prescribe a fixed ratio between N and D, but suggests allocating compute based on the relative efficiency of each factor. In contrast, the Chinchilla scaling law (Hoffmann et al. [34]) combines N and D into a single loss function and optimizes their allocation under a compute budget constraint (C ≈ 6ND), deriving specific optimal values (Nₒₚₜ, Dₒₚₜ) based on coefficients α and β. Thus, KM allows flexible allocation based on individual factor efficiencies, while Chinchilla prescribes a specific optimal ratio derived from joint optimization [30][34].


In [ ]:
import os
import re
import time
import json
import hashlib
import weaviate
import numpy as np
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=ResourceWarning)

from llama_parse import LlamaParse
from weaviate.classes.config import Configure, Property, DataType
from langchain_core.documents import Document as LC_Document
from langchain_experimental.text_splitter import SemanticChunker
from langchain_ollama import OllamaEmbeddings, OllamaLLM
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import CrossEncoder


from dotenv import load_dotenv
load_dotenv()

FILE_PATH = "./NLP_project/file_survey_paper.pdf"
CACHE_DIR = "./rag_cache"
COLLECTION_NAME = "LlamaParse_MRL_nomic"
TARGET_DIM = 256
EMBED_MODEL_NAME = "nomic-embed-text"
GENERATION_MODEL = "deepseek-r1:8b"
RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"

# Retrieval settings
SEARCH_TOP_K = 30              # Default; overridden in main() via input
RERANK_TOP_N = 5           # Keep top N after re-ranking
MAX_CHUNK_CHARS = 1500     # Truncate chunks before sending to LLM

# Generation settings
LLM_NUM_CTX = 4096         # Context window size
LLM_NUM_PREDICT = 1024     # Max output tokens
LLM_TEMPERATURE = 0.1      # Low = less wandering = faster



# UTILITIES
class Timer:
    """Simple context-manager timer for profiling each step."""
    def __init__(self, label: str):
        self.label = label
    def __enter__(self):
        self.start = time.perf_counter()
        print(f"\n  [{self.label}] Starting...")
        return self
    def __exit__(self, *args):
        self.elapsed = time.perf_counter() - self.start
        print(f" [{self.label}] Done in {self.elapsed:.1f}s")


def file_hash(path: str) -> str:
    """Compute MD5 hash of a file for cache invalidation."""
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(8192), b""):
            h.update(block)
    return h.hexdigest()


def get_cache_path(pdf_path: str) -> str:
    """Return cache file path based on the PDF filename."""
    os.makedirs(CACHE_DIR, exist_ok=True)
    base = os.path.splitext(os.path.basename(pdf_path))[0]
    return os.path.join(CACHE_DIR, f"{base}_cache.json")



# STEP 1: PARSE + CHUNK + EMBED (with caching)
def parse_and_chunk(pdf_path: str) -> list[LC_Document]:
    """Parse PDF with LlamaParse → pre-split → semantic chunk."""
    api_key = os.getenv("LLAMA_CLOUD_API_KEY")
    parser = LlamaParse(
        api_key=api_key,
        result_type="markdown",
        num_workers=4,        # Increase parallelism for large docs
        verbose=True,
        language="en",
    )

    print(f"Parsing: {pdf_path}")
    documents = parser.load_data(pdf_path)
    print(f"Parsed {len(documents)} document sections.")

    # Convert LlamaIndex → LangChain documents with metadata
    # NOTE: LlamaParse returns empty metadata (no page_label).
    # Since it returns ~1 section per page, we use the index as page number.
    lc_docs = []
    for idx, doc in enumerate(documents):
        page_num = doc.metadata.get("page_label", str(idx + 1))
        lc_docs.append(
            LC_Document(
                page_content=doc.text,
                metadata={
                    "source": doc.metadata.get("file_name", os.path.basename(pdf_path)),
                    "page_number": page_num,
                },
            )
        )

    # Pre-split to stay within embedding model token limits
    pre_splitter = RecursiveCharacterTextSplitter(
        chunk_size=4000,
        chunk_overlap=400,
    )
    pre_split_docs = pre_splitter.split_documents(lc_docs)
    print(f"Pre-split into {len(pre_split_docs)} chunks.")

    # Semantic chunking
    embed_model = OllamaEmbeddings(model=EMBED_MODEL_NAME)
    semantic_chunker = SemanticChunker(
        embed_model,
        breakpoint_threshold_type="percentile",
        breakpoint_threshold_amount=95.0,
    )
    semantic_chunks = semantic_chunker.split_documents(pre_split_docs)
    print(f"Semantic chunking produced {len(semantic_chunks)} chunks.")

    return semantic_chunks


def batch_embed_and_slice(chunks: list[LC_Document]) -> list[dict]:
    """
    Batch-embed all chunks in a SINGLE call, apply MRL slicing + normalization.
    This replaces TWO separate per-chunk embed_query() loops.
    """
    embed_model = OllamaEmbeddings(model=EMBED_MODEL_NAME)

    # Extract all texts for batch embedding
    all_texts = [chunk.page_content for chunk in chunks]

    print(f"Batch-embedding {len(all_texts)} chunks with {EMBED_MODEL_NAME}...")
    all_vectors = embed_model.embed_documents(all_texts)  # Single batched call!

    # MRL slicing + normalization (vectorized with NumPy)
    vectors_np = np.array(all_vectors, dtype=np.float32)[:, :TARGET_DIM]
    norms = np.linalg.norm(vectors_np, axis=1, keepdims=True)
    norms[norms == 0] = 1  # Avoid division by zero
    vectors_np = vectors_np / norms

    print(f"MRL: Sliced to {TARGET_DIM} dims and normalized.")

    # Build processed chunks with metadata
    processed = []
    for i, chunk in enumerate(chunks):
        processed.append({
            "content": chunk.page_content,
            "metadata": chunk.metadata,
            "vector": vectors_np[i].tolist(),
        })

    return processed


def load_or_process(pdf_path: str) -> list[dict]:
    """
    Main caching layer. Returns processed chunks with vectors.
    If cache exists and PDF hasn't changed, loads from cache.
    Otherwise, parses → chunks → embeds → caches.
    """
    cache_path = get_cache_path(pdf_path)
    current_hash = file_hash(pdf_path)

    # Check cache
    if os.path.exists(cache_path):
        with open(cache_path, "r", encoding="utf-8") as f:
            cache_data = json.load(f)

        if cache_data.get("file_hash") == current_hash:
            chunks = cache_data["chunks"]
            print(f"Cache hit! Loaded {len(chunks)} pre-computed chunks.")
            return chunks
        else:
            print("PDF changed — re-processing...")

    # Cache miss: full pipeline
    with Timer("Parse + Chunk"):
        semantic_chunks = parse_and_chunk(pdf_path)

    with Timer("Batch Embed + MRL"):
        processed_chunks = batch_embed_and_slice(semantic_chunks)

    # Save to cache (with hash for invalidation)
    cache_data = {
        "file_hash": current_hash,
        "file_path": pdf_path,
        "target_dim": TARGET_DIM,
        "num_chunks": len(processed_chunks),
        "chunks": processed_chunks,
    }
    with open(cache_path, "w", encoding="utf-8") as f:
        json.dump(cache_data, f, ensure_ascii=False)
    print(f"Cached {len(processed_chunks)} chunks to {cache_path}")

    return processed_chunks

# STEP 2: WEAVIATE UPLOAD (using pre-computed vectors)
def upload_to_weaviate(processed_chunks: list[dict]):
    """Upload chunks with pre-computed MRL vectors. No re-embedding!"""
    client = weaviate.connect_to_local()

    try:
        # Re-create collection for fresh schema
        if client.collections.exists(COLLECTION_NAME):
            client.collections.delete(COLLECTION_NAME)

        client.collections.create(
            name=COLLECTION_NAME,
            vectorizer_config=Configure.Vectorizer.none(),
            properties=[
                Property(name="content", data_type=DataType.TEXT),
                Property(name="source", data_type=DataType.TEXT),
                Property(name="page_number", data_type=DataType.TEXT),
            ],
        )

        collection = client.collections.get(COLLECTION_NAME)

        # Batch upload with PRE-COMPUTED vectors (no embed_query calls!)
        with collection.batch.dynamic() as batch:
            for item in processed_chunks:
                page_val = str(
                    item["metadata"].get(
                        "page_label",
                        item["metadata"].get("page_number", "Unknown"),
                    )
                )
                batch.add_object(
                    properties={
                        "content": item["content"],
                        "source": str(item["metadata"].get("source", "Unknown")),
                        "page_number": page_val,
                    },
                    vector=item["vector"],
                )

        print(f"Uploaded {len(processed_chunks)} chunks to Weaviate ({COLLECTION_NAME}).")

    finally:
        client.close()


# STEP 3: QUERY EMBEDDING
def embed_query(query_text: str) -> list[float]:
    """Embed a user query, apply MRL slicing + normalization."""
    embed_model = OllamaEmbeddings(model=EMBED_MODEL_NAME)
    full_vector = embed_model.embed_query(query_text)

    # MRL slice + normalize
    sliced = np.array(full_vector[:TARGET_DIM], dtype=np.float32)
    norm = np.linalg.norm(sliced)
    if norm > 0:
        sliced = sliced / norm

    return sliced.tolist()


# STEP 4: WEAVIATE VECTOR SEARCH
def weaviate_search(query_vec: list[float], top_k: int = SEARCH_TOP_K) -> list[dict]:
    """
    Search Weaviate using near_vector with pre-computed MRL vectors.
    Weaviate uses HNSW internally — O(log n) search, scales to millions.
    """
    client = weaviate.connect_to_local()

    try:
        collection = client.collections.get(COLLECTION_NAME)

        response = collection.query.near_vector(
            near_vector=query_vec,
            limit=top_k,
            return_properties=["content", "page_number", "source"],
        )

        results = []
        for obj in response.objects:
            results.append({
                "content": obj.properties["content"],
                "page": obj.properties.get("page_number", "Unknown"),
                "source": obj.properties.get("source", "Unknown"),
            })

        print(f"Weaviate returned {len(results)} results.")
        return results

    finally:
        client.close()


# STEP 5: CROSS-ENCODER RE-RANKING (no LLM distillation!)
def rerank_chunks(query: str, chunks: list[dict], top_n: int = RERANK_TOP_N) -> list[dict]:
    """
    Re-rank retrieved chunks using a cross-encoder.
    This REPLACES the old LLM distillation step (gemma3:4b) entirely.
    Cross-encoders are purpose-built for relevance scoring — faster + better.
    """
    reranker = CrossEncoder(RERANKER_MODEL)

    # Prepare query-document pairs
    pairs = [[query, chunk["content"]] for chunk in chunks]

    # Score all pairs
    scores = reranker.predict(pairs)

    # Attach scores and sort
    for i, score in enumerate(scores):
        chunks[i]["rerank_score"] = float(score)

    ranked = sorted(chunks, key=lambda x: x["rerank_score"], reverse=True)

    print(f"Re-ranked {len(chunks)} → keeping top {top_n}")
    for i, c in enumerate(ranked[:top_n]):
        print(f"   Rank {i+1}: Page {c['page']} | Score: {c['rerank_score']:.4f}")

    return ranked[:top_n]


# STEP 6: PROMPT SYNTHESIS
def build_prompt(query: str, top_chunks: list[dict]) -> str:
    """
    Build a grounded prompt with truncated context.
    MAX_CHUNK_CHARS prevents context explosion for large documents.
    """
    system_rules = (
        "You are a professional research assistant. "
        "Use ONLY the provided context to answer the question.\n"
        "If the answer is not in the context, say you don't know.\n"
        "For every fact, cite the page number as [Page X].\n"
        "Be concise and direct.\n\n"
        "CONSTRAINTS:\n"
        "- DO NOT use any outside knowledge or training data. Only use the CONTEXT below.\n"
        "- DO NOT speculate, assume, or infer beyond what is explicitly stated.\n"
        "- DO NOT repeat the question or paraphrase it back.\n"
        "- DO NOT add disclaimers like 'based on the context provided'.\n"
        "- DO NOT produce long chain-of-thought reasoning. Give the answer directly.\n"
        "- If multiple pages support a fact, cite all of them."
    )

    context_block = ""
    for i, chunk in enumerate(top_chunks):
        # Truncate each chunk to prevent context explosion
        content = chunk["content"][:MAX_CHUNK_CHARS]
        context_block += f"\n--- CHUNK {i+1} (PAGE {chunk['page']}) ---\n{content}\n"

    prompt = f"""{system_rules}

CONTEXT:
{context_block}

USER QUESTION:
{query}

GROUNDED ANSWER:"""

    return prompt



# STEP 7: LLM GENERATION
def generate_answer(prompt: str) -> str:
    """Generate a grounded answer using DeepSeek-R1:8b with capped parameters."""
    llm = OllamaLLM(
        model=GENERATION_MODEL,
        num_ctx=LLM_NUM_CTX,
        num_predict=LLM_NUM_PREDICT,
        temperature=LLM_TEMPERATURE,
    )

    print(f"{GENERATION_MODEL} is generating...")
    raw_response = llm.invoke(prompt)

    # DeepSeek-R1 wraps reasoning in <think>...</think> tags.
    # Strip them to get only the visible answer.
    answer = re.sub(r"<think>.*?</think>", "", raw_response, flags=re.DOTALL).strip()

    if not answer:
        # If stripping left nothing, the model only produced thinking — return raw
        print("[WARNING] Model produced only <think> content. Returning raw response.")
        return raw_response

    return answer



# MAIN PIPELINE
def main():
    global SEARCH_TOP_K
    SEARCH_TOP_K = int(input("How many chunks to retrieve? [default=30]: ") or 30)

    timings = {}
    total_start = time.perf_counter()

    # ── Step 1: Parse + Embed (cached) ──
    with Timer("Parse + Embed (cached)") as t:
        processed_chunks = load_or_process(FILE_PATH)
    timings["1. Parse + Embed"] = t.elapsed

    # ── Step 2: Upload to Weaviate ──
    with Timer("Weaviate Upload") as t:
        upload_to_weaviate(processed_chunks)
    timings["2. Weaviate Upload"] = t.elapsed

    # ── Step 3: User Query + Embedding ──
    query_text = input("\nEnter your question: ")

    with Timer("Query Embedding") as t:
        query_vector = embed_query(query_text)
    timings["3. Query Embed"] = t.elapsed

    # ── Step 4: Weaviate Search ──
    with Timer("Weaviate Search") as t:
        retrieved_chunks = weaviate_search(query_vector, top_k=SEARCH_TOP_K)
    timings["4. Weaviate Search"] = t.elapsed

    # ── Step 5: Re-rank ──
    with Timer("Cross-Encoder Re-ranking") as t:
        top_chunks = rerank_chunks(query_text, retrieved_chunks, top_n=RERANK_TOP_N)
    timings["5. Re-ranking"] = t.elapsed

    # ── Step 6: Prompt Synthesis ──
    with Timer("Prompt Synthesis") as t:
        final_prompt = build_prompt(query_text, top_chunks)
    timings["6. Prompt Synthesis"] = t.elapsed

    # ── Step 7: LLM Generation ──
    with Timer(f"LLM Generation ({GENERATION_MODEL})") as t:
        answer = generate_answer(final_prompt)
    timings["7. LLM Generation"] = t.elapsed

    # ── Output ──
    total_elapsed = time.perf_counter() - total_start

    print("\n" + "=" * 60)
    print("FINAL RESPONSE:")
    print("=" * 60)
    print(answer)
    print("=" * 60)

    # Performance summary
    print("\n PERFORMANCE SUMMARY:")
    print("-" * 40)
    for step, elapsed in timings.items():
        bar = "█" * int(elapsed / total_elapsed * 30)
        print(f"  {step:<25} {elapsed:>6.1f}s  {bar}")
    print("-" * 40)
    print(f"  {'TOTAL':<25} {total_elapsed:>6.1f}s")
    print()


if __name__ == "__main__":
    main()